In [ ]:
# -*- coding: utf-8 -*-
"""
HWSD2 → 0.1° grid of D1 (0-20 cm) soil properties

"""

import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import re
import rioxarray as rxr
from rasterio.enums import Resampling
import pyodbc
from shapely import unary_union
from shapely.geometry import mapping
from shapely.geometry import Polygon



In [ ]:
# ============== CONFIG ==============
AOI_PATH = r"..\..\ExtractedDatasets\GeoBoundaries\AOI_DZA_TUN\aoi_dza_tun.geojson"
RASTER   = r"..\..\Datasets\SoilDataset\HWSD2_RASTER\HWSD2.bil"
MDB      = r"..\..\Datasets\SoilDataset\HWSD2_DB\HWSD2.mdb"

# Output
OUT_GRID_DEBUG   = r"..\..\ExtractedDatasets\SoilFeatures\hwsd2_grid01_debug.csv"
OUT_FINAL_CSV    = r"..\..\ExtractedDatasets\SoilFeatures\hwsd2_grid01_D1.csv"

# Tables & columns (change if your MDB uses different names)
T_COMPONENTS     = "HWSD2_COMPONENTS"   # must have MU_GLOBAL, SU_CODE, SHARE
T_LAYERS         = "HWSD2_LAYERS"       # must have SU_CODE, LAYER, and features below

COL_MU           = "MU_GLOBAL"
COL_SU           = "SU_CODE"
COL_SHARE        = "SHARE"              # component proportion within MU (0..100 or 0..1)
COL_LAYER        = "LAYER"              # 'D1', 'D2', ...

# Requested feature columns (exact names expected in HWSD2_LAYERS)
FEATURE_COLS = [
    "COARSE", "SAND", "SILT", "CLAY",
    "TEXTURE_USDA", "TEXTURE_SOTER",
    "BULK", "REF_BULK", "ORG_CARBON", "PH_WATER",
    "TOTAL_N", "CN_RATIO",
    "CEC_SOIL", "CEC_CLAY", "CEC_EFF",
    "TEB", "BSAT", "ALUM_SAT", "ESP",
    "TCARBON_EQ", "GYPSUM", "ELEC_COND"
]
CATEGORICAL_COLS = ["TEXTURE_USDA", "TEXTURE_SOTER"]
NUMERIC_COLS     = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

# Regridding resolution (degrees)
TARGET_RES = (0.1, 0.1)

# ====================================


def ensure_dir(p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)




In [ ]:
# ---------- Step 1: Build 0.1° MU grid over AOI ----------
print("Step 1/4: Building 0.1° MU grid...")

# 1) Read AOI and set CRS explicitly if missing, then work in EPSG:4326
aoi = gpd.read_file(AOI_PATH)

# If file has no CRS, set it (CHANGE this if your AOI isn't WGS84!)
if aoi.crs is None:
    aoi = aoi.set_crs(4326)
aoi = aoi.to_crs(4326)

# 2) Fix any invalid geometries (buffer(0) trick)
aoi["geometry"] = aoi.geometry.buffer(0)

# 3) Dissolve to a single MultiPolygon (optional but safest for clip)
aoi_union = unary_union(aoi.geometry)
geoms = [mapping(aoi_union)]  # <-- rio.clip expects list of GeoJSON-like geometries

# 4) Read MU raster and set CRS if missing (HWSD2.prj usually provides it)
da = rxr.open_rasterio(RASTER, masked=True).squeeze()
if da.rio.crs is None:
    da = da.rio.write_crs("EPSG:4326", inplace=False)

# 5) Clip using the explicit geometry list (avoid passing a full FeatureCollection)
da_clip = da.rio.clip(geoms, aoi.crs, drop=True)

# 6) Reproject/aggregate to 0.1° (dominant MU per cell)
da_01 = da_clip.rio.reproject(
    dst_crs="EPSG:4326",
    resolution=(0.1, 0.1),
    resampling=Resampling.mode,
)

# 7) Turn into DataFrame: x(lon), y(lat), MU_GLOBAL
df_grid = da_01.to_dataframe(name=COL_MU).reset_index()
df_grid = df_grid[df_grid[COL_MU].notna()].copy()
df_grid[COL_MU] = df_grid[COL_MU].astype("int64")

ensure_dir(OUT_GRID_DEBUG)
df_grid.to_csv(OUT_GRID_DEBUG, index=False)
print(f"  Grid preview saved: {OUT_GRID_DEBUG}")
print(f"  Grid cells: {len(df_grid):,}")




In [ ]:
# ---------- Step 2 & 3 (v2 schema): Read HWSD2_SMU / HWSD2_LAYERS (D1) and join by SMU id ----------
print("Step 2/4: Reading MDB tables (v2: SMU-level, no components)...")

def open_mdb_conn(mdb_path):
    pyodbc.pooling = False
    conn = pyodbc.connect(
        r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=" + mdb_path + ";"
    )
    try:
        conn.setencoding('utf-8')
        conn.setdecoding(pyodbc.SQL_CHAR, 'utf-8')
        conn.setdecoding(pyodbc.SQL_WCHAR, 'utf-16le')
        conn.setdecoding(pyodbc.SQL_WMETADATA, 'utf-16le')
    except Exception:
        pass
    return conn

conn = open_mdb_conn(MDB)

# --- Identify SMU key in HWSD2_SMU ---
smu_table = "HWSD2_SMU"
smu_head  = pd.read_sql(f"SELECT TOP 5 * FROM [{smu_table}];", conn)
smu_colsL = {c.lower(): c for c in smu_head.columns}

# common candidates for the MU/SMU id in v2
mu_key_candidates = ["hwsd2_smu_id", "smu_id", "mu_global", "mukey", "mu"]
COL_MU_SMU = None
for k in mu_key_candidates:
    if k in smu_colsL:
        COL_MU_SMU = smu_colsL[k]
        break
if COL_MU_SMU is None:
    # try substring contains
    for low, orig in smu_colsL.items():
        if any(k in low for k in mu_key_candidates):
            COL_MU_SMU = orig
            break
if COL_MU_SMU is None:
    raise RuntimeError(f"Could not detect MU/SMU key in {smu_table}. Columns: {list(smu_head.columns)}")

print(f"  SMU key in {smu_table}: {COL_MU_SMU}")

# --- Identify SMU key + LAYER in HWSD2_LAYERS ---
layers_table = "HWSD2_LAYERS"
layers_head  = pd.read_sql(f"SELECT TOP 5 * FROM [{layers_table}];", conn)
layers_colsL = {c.lower(): c for c in layers_head.columns}

# LAYER column
COL_LAYER = None
if "layer" in layers_colsL:
    COL_LAYER = layers_colsL["layer"]
else:
    for low, orig in layers_colsL.items():
        if "layer" in low:
            COL_LAYER = orig
            break
if COL_LAYER is None:
    raise RuntimeError(f"Could not detect LAYER column in {layers_table}. Columns: {list(layers_head.columns)}")

# SMU key in layers (usually same name as in SMU table)
COL_MU_LAY = None
if COL_MU_SMU.lower() in layers_colsL:
    COL_MU_LAY = layers_colsL[COL_MU_SMU.lower()]
else:
    for low, orig in layers_colsL.items():
        if low == COL_MU_SMU.lower() or any(k in low for k in mu_key_candidates):
            COL_MU_LAY = orig
            break
if COL_MU_LAY is None:
    raise RuntimeError(f"Could not detect MU/SMU key in {layers_table}. Columns: {list(layers_head.columns)}")

print(f"  LAYERS key in {layers_table}: {COL_MU_LAY}  | LAYER col: {COL_LAYER}")

# --- Map requested feature names to actual columns (handle synonyms) ---
# Build a synonym map to be flexible with v2 naming
synonyms = {
    "BULK": ["bulk", "bulk_density", "bulk density"],
    "REF_BULK": ["ref_bulk", "ref_bulk_density", "ref bulk density", "ref_bulkdensity"],
    "ORG_CARBON": ["org_carbon", "organic_carbon", "org carbon", "org_c", "org-c"],
    "PH_WATER": ["ph_water", "ph(h2o)", "ph_h2o", "ph (water)"],
    "TOTAL_N": ["total_n", "total nitrogen", "n_total"],
    "CN_RATIO": ["cn_ratio", "c/n", "c_n_ratio"],
    "CEC_SOIL": ["cec_soil", "cec soil", "cec"],
    "CEC_CLAY": ["cec_clay", "cec clay"],
    "CEC_EFF": ["cec_eff", "effective_cec", "cec(eff)"],
    "TEB": ["teb", "total_exchangeable_bases"],
    "BSAT": ["bsat", "base_saturation", "base sat"],
    "ALUM_SAT": ["alum_sat", "aluminium_saturation", "aluminum_saturation"],
    "ESP": ["esp", "exchangeable_sodium_percentage"],
    "TCARBON_EQ": ["tcarbon_eq", "total_carbonate_equiv", "tcarbonate_eq", "tcarbon equivalent"],
    "GYPSUM": ["gypsum"],
    "ELEC_COND": ["elec_cond", "electrical_conductivity", "ec"],
    "COARSE": ["coarse", "coarse_fragments"],
    "SAND": ["sand"],
    "SILT": ["silt"],
    "CLAY": ["clay"],
    "TEXTURE_USDA": ["texture_usda", "usda_texture"],
    "TEXTURE_SOTER": ["texture_soter", "soter_texture"],
}

feature_map = {}
for wanted in FEATURE_COLS:
    wl = wanted.lower()
    # exact
    if wl in layers_colsL:
        feature_map[wanted] = layers_colsL[wl]
        continue
    # try synonyms
    matched = None
    for syn in synonyms.get(wanted, []):
        if syn in layers_colsL:
            matched = layers_colsL[syn]
            break
        # substring
        for low, orig in layers_colsL.items():
            if syn.replace(" ", "_") in low.replace(" ", "_"):
                matched = orig
                break
        if matched:
            break
    # generic substring fallback by base token
    if not matched:
        base = wl.replace("_", "")
        for low, orig in layers_colsL.items():
            if base in low.replace("_", ""):
                matched = orig
                break
    if matched:
        feature_map[wanted] = matched

missing = [f for f in FEATURE_COLS if f not in feature_map]
if missing:
    print("  Missing features in HWSD2_LAYERS (will be NA after merge):", ", ".join(missing))

# --- Load D1 only with resolved names ---
sel_cols_layers = [f"[{COL_MU_LAY}]", f"[{COL_LAYER}]"] + [f"[{feature_map[f]}]" for f in feature_map]
q_layers = f"SELECT {', '.join(sel_cols_layers)} FROM [{layers_table}] WHERE [{COL_LAYER}]='D1';"
layers_d1 = pd.read_sql(q_layers, conn)

conn.close()
print(f"  Loaded D1 rows: {len(layers_d1):,}")

# --- Step 3: prepare MU-level properties (already SMU-level → no weighting needed) ---
# Rename to standard names expected downstream
layers_d1 = layers_d1.rename(columns={COL_MU_LAY: COL_MU, COL_LAYER: "LAYER"})
# Keep only MU + features
keep_cols = [COL_MU] + [feature_map[f] for f in feature_map]
layers_d1 = layers_d1[keep_cols].drop_duplicates(subset=[COL_MU])

# Rename feature columns back to requested canonical names
inv_map = {v: k for k, v in feature_map.items()}
layers_d1 = layers_d1.rename(columns=inv_map)

# If your grid MU col is named differently, make sure df_grid[COL_MU] exists
if COL_MU not in df_grid.columns:
    # In your grid it’s likely MU_GLOBAL; align name
    if "MU_GLOBAL" in df_grid.columns and df_grid["MU_GLOBAL"].dtype.kind in "iu":
        df_grid = df_grid.rename(columns={"MU_GLOBAL": COL_MU})
    else:
        raise RuntimeError(f"Grid MU column not found. Have: {list(df_grid.columns)} | Expected MU key: {COL_MU}")

print("Step 3/4: MU-level D1 properties ready (no component aggregation required).")
print(f"  Columns prepared: {list(layers_d1.columns)}")


In [ ]:
# ---------- Step 4/4: Join to 0.1° grid & save ----------
print("Step 4/4: Joining to 0.1° grid & saving...")

# Robustly detect MU/SMU key in both grid and layers and align to a canonical name
candidates = [COL_MU_SMU, COL_MU, COL_MU_LAY, "HWSD2_SMU_ID", "MU_GLOBAL"]
df_key = next((c for c in candidates if c in df_grid.columns), None)
layers_key = next((c for c in candidates if c in layers_d1.columns), None)

if df_key is None:
    raise RuntimeError(f"Grid MU column not found. Grid has: {list(df_grid.columns)} | tried: {candidates}")
if layers_key is None:
    raise RuntimeError(f"Layers MU column not found. layers_d1 has: {list(layers_d1.columns)} | tried: {candidates}")

# Rename both sides to a canonical merge key 'HWSD2_SMU_ID'
if df_key != "HWSD2_SMU_ID":
    df_grid = df_grid.rename(columns={df_key: "HWSD2_SMU_ID"})
if layers_key != "HWSD2_SMU_ID":
    layers_d1 = layers_d1.rename(columns={layers_key: "HWSD2_SMU_ID"})

# Make sure join keys have same dtype
df_grid["HWSD2_SMU_ID"]   = pd.to_numeric(df_grid["HWSD2_SMU_ID"], errors="coerce").astype("Int64")
layers_d1["HWSD2_SMU_ID"] = pd.to_numeric(layers_d1["HWSD2_SMU_ID"], errors="coerce").astype("Int64")

# Merge
df = df_grid.merge(layers_d1, on="HWSD2_SMU_ID", how="left")

# Optional: cast numeric columns cleanly
num_cols = [c for c in df.columns if c in [
    "COARSE","SAND","SILT","CLAY","BULK","REF_BULK","ORG_CARBON","PH_WATER",
    "TOTAL_N","CN_RATIO","CEC_SOIL","CEC_CLAY","CEC_EFF","TEB","BSAT",
    "ALUM_SAT","ESP","TCARBON_EQ","GYPSUM","ELEC_COND"
]]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Quick sanity checks
total_cells   = len(df)
matched_cells = df["COARSE"].notna().sum()  # any D1 field as proxy
print(f"  Cells total: {total_cells:,} | with D1 data: {matched_cells:,} ({matched_cells/total_cells:.1%})")

# Save
ensure_dir(OUT_FINAL_CSV)
df.to_csv(OUT_FINAL_CSV, index=False)
print("Done.")
print(f"Saved: {OUT_FINAL_CSV}")

# # (Optional) also save a compact parquet
# OUT_PARQUET = OUT_FINAL_CSV.replace(".csv", ".parquet")
# df.to_parquet(OUT_PARQUET, index=False)
# print(f"Also saved: {OUT_PARQUET}")


In [ ]:

# df has columns: x (lon), y (lat) = cell centers at 0.1°
res = 0.1
def cell_poly(x, y, r=res):
    return Polygon([(x-r/2, y-r/2), (x+r/2, y-r/2), (x+r/2, y+r/2), (x-r/2, y+r/2)])

gdf = gpd.GeoDataFrame(
    df,
    geometry=[cell_poly(x, y) for x, y in zip(df["x"], df["y"])],
    crs="EPSG:4326"
)

# Save as GeoPackage for GIS
gpkg_out = OUT_FINAL_CSV.replace(".csv", ".gpkg")
gdf.to_file(gpkg_out, driver="GPKG", layer="hwsd2_d1_grid01")
print("Saved GeoPackage:", gpkg_out)


In [ ]:
num_cols = ["COARSE","SAND","SILT","CLAY","BULK","REF_BULK","ORG_CARBON","PH_WATER",
            "TOTAL_N","CN_RATIO","CEC_SOIL","CEC_CLAY","CEC_EFF",
            "TEB","BSAT","ALUM_SAT","ESP","TCARBON_EQ","GYPSUM","ELEC_COND"]

print(df[num_cols].describe().T.round(3))
print("Missing % per column:\n", (df[num_cols].isna().mean()*100).round(2))

In [ ]:
fig = plt.figure(figsize=(8,6))
gdf.plot(column="PH_WATER", legend=True)
plt.title("HWSD2 D1 pH (water) – 0.1°")
plt.show()
